# Capítulo 2 — Envenenamento de dados e portas dos fundos

## Missão
O ataque mais silencioso não mexe na entrada: mexe nos **dados de treino**. Você vai:
- contaminar rótulos ao acaso e de forma direcionada (como no ataque ao SVM do cap. 11);
- medir o estrago;
- plantar uma **porta dos fundos** (*backdoor*): um pequeno padrão de pixels que faz o modelo responder o que o atacante quer, sem perder acurácia nos dados limpos. Por isso ela é tão difícil de notar.

**No fim deste caderno você terá:** o módulo `meu_escudo/envenenamento.py` (inversão de rótulos aleatória e direcionada e o gatilho de backdoor) e as medidas de dano de cada ataque.

**Leitura essencial (opcional, livro em inglês):**
- 1.5 Data Poisoning Attacks · PDF p. 37
- 11.2 SVM Under Evasion Attacks · PDF p. 165
- 11.4 Adversarial Label Contamination on SVM · PDF p. 167

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap02_envenenamento_de_dados_e_portas_dos_fundos.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados fictícios do projeto)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_escudo/
os.makedirs("../meu_escudo", exist_ok=True)
open("../meu_escudo/__init__.py", "a").close()        # meu_escudo/ é o projeto que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
import warnings; warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: meu_escudo")

## Ideia em 30 segundos
- **Disponibilidade:** quanto pior o modelo, melhor para o atacante. Inverter rótulos de exemplos **perto da fronteira** estraga mais que inverter ao acaso.
- **Integridade (backdoor):** treina-se com algumas imagens que levam um **gatilho**, por exemplo um quadradinho no canto, rotuladas com a classe-alvo. O modelo aprende "gatilho → alvo" e continua perfeito no resto. As métricas usuais não acusam nada.

### 2.1 **O envenenamento.** Crie `envenenamento.py` com:
- `inverter_aleatorio(y, frac, n_classes, semente)`: troca o rótulo de uma fração por outra classe qualquer;
- `inverter_direcionado(X, y, modelo, frac)`: troca os rótulos dos exemplos de **menor margem** (a menor diferença entre as duas maiores probabilidades) pela segunda classe mais provável;
- `com_gatilho(X)`: acende os 4 pixels do canto inferior direito (índices 54, 55, 62 e 63);
- `envenenar_backdoor(X, y, alvo, frac, semente)`: devolve `X` e `y` com uma fração de exemplos extras com gatilho e rótulo `alvo`.

_A célula grava `meu_escudo/envenenamento.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_escudo/envenenamento.py


### 2.2 **Inversão aleatória × direcionada.** Com os dígitos (70/30), treine um `SVC(probability=True)` limpo e meça a acurácia. Depois, para frações de 5%, 10%, 20% e 30%, treine com rótulos invertidos ao acaso e de forma direcionada (usando o modelo limpo para achar as menores margens) e compare a acurácia no teste **limpo**.

### 2.3 **A porta dos fundos.** Envenene o treino com 5% de cópias com gatilho rotuladas como `7`. Treine um `SVC` e meça:
- (a) a acurácia no teste limpo, que deve ficar quase igual à do modelo limpo;
- (b) a **taxa de sucesso do ataque**: a fração dos dígitos de teste que **não** são 7 e viram 7 quando recebem o gatilho.

Compare com o modelo limpo recebendo o gatilho.

### 2.4 **Quanto veneno basta?** Varie a fração do backdoor (0,5%, 1%, 2%, 5% e 10%) e desenhe, no mesmo gráfico, a acurácia limpa e a taxa de sucesso do gatilho. Com que fração o ataque já funciona?

### 2.5 **Evasão contra o SVM linear (cap. 11).** Num SVM linear binário (dígito 1 contra o 7), o atacante desloca o exemplo na direção de `-w` até cruzar a fronteira. Calcule, para cada 7 do teste, a menor perturbação L2 que o faz virar 1 (`margem / ‖w‖`) e mostre a mediana. Compare com a distância típica entre dois dígitos.

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
